In [10]:
# --- Imports ---
import math
import re
import string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

# --- Text Preprocessing Function ---
def preprocess_text(text):
    """
    Lowercase, remove punctuation and extra spaces.
    """
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [11]:
# Documents
documents = [
    "d1- Genshin Impact takes place in the fantasy world of Teyvat, home to seven nations, each of which is tied to a different element and ruled by a different god called an Archon. The story follows the Traveler, an interstellar adventurer who, at the start of the game, is separated from their twin sibling after the two land in Teyvat. Thereafter, the Traveler journeys across the nations of Teyvat in search of the lost sibling, accompanied by their guide, Paimon. Along the way, the two befriend myriad individuals, become involved in the affairs of its nations, and begin to unravel the mysteries of the land.",
    "d2- Development began in 2017 and takes inspiration from a variety of sources, including The Legend of Zelda: Breath of the Wild, anime, Gnosticism, and an array of cultures and world mythologies.",
    "d3- Genshin Impact has received generally positive reviews, with critics writing approving of its combat mechanics and its immersive open world.",
    "d4- Some criticism has been directed at its simplistic endgame and its gacha-based monetization model.",
    "d5- The game has also been subjected to controversy over censorship of content related to Chinese politics, allegations of colorism in character design, and privacy and security concerns.",
    "d6- Across all platforms, the game is estimated to have grossed nearly $3.8 billion by the end of 2022, representing the highest ever first-year launch revenue for any video game."
]

# Queries
queries = [
    "genshin impact story and development"
]


In [12]:
# Preprocess documents and queries
preprocessed_documents = [preprocess_text(doc) for doc in documents]
preprocessed_queries = [preprocess_text(query) for query in queries]

print("Preprocessed Documents:")
for i, doc in enumerate(preprocessed_documents):
    print(f"Document {i+1}: {doc}")

print("\nPreprocessed Queries:")
for i, query in enumerate(preprocessed_queries):
    print(f"Query {i+1}: {query}")


Preprocessed Documents:
Document 1: d1 genshin impact takes place in the fantasy world of teyvat home to seven nations each of which is tied to a different element and ruled by a different god called an archon the story follows the traveler an interstellar adventurer who at the start of the game is separated from their twin sibling after the two land in teyvat thereafter the traveler journeys across the nations of teyvat in search of the lost sibling accompanied by their guide paimon along the way the two befriend myriad individuals become involved in the affairs of its nations and begin to unravel the mysteries of the land
Document 2: d2 development began in 2017 and takes inspiration from a variety of sources including the legend of zelda breath of the wild anime gnosticism and an array of cultures and world mythologies
Document 3: d3 genshin impact has received generally positive reviews with critics writing approving of its combat mechanics and its immersive open world
Document 4: 

In [13]:
# TF-IDF vectorization
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(preprocessed_documents)

vocabulary = tfidf_vectorizer.get_feature_names_out()
print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
print("Vocabulary Size:", len(vocabulary))


TF-IDF Matrix Shape: (6, 143)
Vocabulary Size: 143


In [14]:
# Assume query 0 is most relevant to document 1 (story + development)
simulated_relevance = {
    0: [0, 1]  # d1 and d2 relevant to query 0
}

print("Simulated Relevance Judgments:")
for query_index, relevant_docs in simulated_relevance.items():
    print(f"Query \"{queries[query_index]}\": Relevant Documents - {relevant_docs}")


Simulated Relevance Judgments:
Query "genshin impact story and development": Relevant Documents - [0, 1]


In [16]:
vocab_dict = {term: index for index, term in enumerate(vocabulary)}
vocab_size = len(vocabulary)
alpha = 1

term_doc_counts_relevant = {}
term_doc_counts_non_relevant = {}

for query_index, relevant_doc_indices in simulated_relevance.items():
    non_relevant_doc_indices = [i for i in range(len(documents)) if i not in relevant_doc_indices]

    term_counts_relevant = {term: 0 for term in vocabulary}
    term_counts_non_relevant = {term: 0 for term in vocabulary}

    for doc_index in relevant_doc_indices:
        for term in preprocessed_documents[doc_index].split():
            if term in vocabulary: # Check if term is in vocabulary
                term_counts_relevant[term] += 1

    for doc_index in non_relevant_doc_indices:
        for term in preprocessed_documents[doc_index].split():
            if term in vocabulary: # Check if term is in vocabulary
                term_counts_non_relevant[term] += 1


    term_doc_counts_relevant[query_index] = term_counts_relevant
    term_doc_counts_non_relevant[query_index] = term_counts_non_relevant

prob_k_given_relevant = {}
prob_k_given_non_relevant = {}

for query_index in simulated_relevance.keys():
    total_relevant_terms = sum(term_doc_counts_relevant[query_index].values())
    total_non_relevant_terms = sum(term_doc_counts_non_relevant[query_index].values())

    prob_k_given_relevant[query_index] = {}
    prob_k_given_non_relevant[query_index] = {}

    for term in vocabulary:
        prob_k_given_relevant[query_index][term] = (term_doc_counts_relevant[query_index][term] + alpha) / (total_relevant_terms + alpha * vocab_size)
        prob_k_given_non_relevant[query_index][term] = (term_doc_counts_non_relevant[query_index][term] + alpha) / (total_non_relevant_terms + alpha * vocab_size)

print("Initial Probabilities calculated.")

Initial Probabilities calculated.


In [17]:
reweighted_query_weights = {}

for query_index, preprocessed_query in enumerate(preprocessed_queries):
    reweighted_weights = {}
    prob_r = prob_k_given_relevant[query_index]
    prob_nr = prob_k_given_non_relevant[query_index]

    for term in preprocessed_query.split():
        if term in prob_r and term in prob_nr:
            p_k_r = prob_r[term]
            p_k_nr = prob_nr[term]

            epsilon = 1e-9
            numerator = p_k_r * (1 - p_k_nr) + epsilon
            denominator = (1 - p_k_r) * p_k_nr + epsilon
            weight = math.log(numerator / denominator)
            reweighted_weights[term] = weight
        else:
            reweighted_weights[term] = 0.0

    reweighted_query_weights[query_index] = reweighted_weights

print("Reweighted Query Term Weights Calculated.")


Reweighted Query Term Weights Calculated.


In [18]:
reweighted_ranked_document_indices = []
reweighted_similarity_scores = []

for query_index, preprocessed_query in enumerate(preprocessed_queries):
    weights_dict = reweighted_query_weights[query_index]
    reweighted_query_vector = [0.0] * vocab_size

    for term, weight in weights_dict.items():
        if term in vocab_dict:
            term_index = vocab_dict[term]
            reweighted_query_vector[term_index] = weight

    reweighted_query_vector_sparse = csr_matrix([reweighted_query_vector])
    similarity_scores = cosine_similarity(reweighted_query_vector_sparse, tfidf_matrix).flatten()
    ranked_document_indices = similarity_scores.argsort()[::-1]

    reweighted_ranked_document_indices.append(ranked_document_indices)
    reweighted_similarity_scores.append(similarity_scores)

    print("Ranked Documents (Reweighted Retrieval):")
    for rank, doc_index in enumerate(ranked_document_indices):
        print(f"Rank {rank+1}: \"{documents[doc_index]}\" (Similarity: {similarity_scores[doc_index]:.4f})")


Ranked Documents (Reweighted Retrieval):
Rank 1: "d2- Development began in 2017 and takes inspiration from a variety of sources, including The Legend of Zelda: Breath of the Wild, anime, Gnosticism, and an array of cultures and world mythologies." (Similarity: 0.1382)
Rank 2: "d1- Genshin Impact takes place in the fantasy world of Teyvat, home to seven nations, each of which is tied to a different element and ruled by a different god called an Archon. The story follows the Traveler, an interstellar adventurer who, at the start of the game, is separated from their twin sibling after the two land in Teyvat. Thereafter, the Traveler journeys across the nations of Teyvat in search of the lost sibling, accompanied by their guide, Paimon. Along the way, the two befriend myriad individuals, become involved in the affairs of its nations, and begin to unravel the mysteries of the land." (Similarity: 0.0282)
Rank 3: "d5- The game has also been subjected to controversy over censorship of content 